In [19]:
import sys
import os
import matplotlib.pyplot as plt

# Add animation script folder to sys.path
sys.path.append(r"C:\Users\chenw\MotionCapturePy")  # where MoCapAnimateAsfAmc.py is

from MoCapAnimateAsfAmc import MotionCapture

import torch
import numpy as np
import glob
from tqdm import tqdm
from Messy_data_Training_conditional_score import ConditionalScoreNet, hyvarinen_loss, compute_hyvarinen_score
from numpy.linalg import inv

In [21]:
# Path to the raw data
data_folder = r"C:\Users\chenw\OneDrive - University of Pittsburgh\_score network code\Markov chain neumeric\CMU motion capture\raw data"
subject = "16"         # Subject ID, e.g., "01"
trial = "46"           # Trial ID, e.g., "01" for 01_01.amc

# Build full file paths
subject_folder = os.path.join(data_folder, f"Subject_{subject}")
asf_file = os.path.join(subject_folder, f"{subject}.asf")
amc_file = os.path.join(subject_folder, f"Trial_{trial}.amc")

# Load motion
motion = MotionCapture(asf_file, amc_file)
# motion.Animate(motion.limbs)

# Extract data: motion.data has shape (num_frames, num_bones, 3)
T, B, _ = motion.data.shape   # T = frames, B = number of bones
X = motion.data.reshape(T, B * 3)  # Flatten each frame

# Done! Now you can use X
print("Shape of X:", X.shape)  # Expect (T, d) where d = 3 * num_bones

# see one piece of data
# print("one piece of X:", X[101])
print("type(X): ", type(X))

running_trials = {
    "02": ["03"],
    "09": [f"{i:02d}" for i in range(1, 12)],  # "01" to "11"
    "16": ["08"] + [f"{i:02d}" for i in range(35, 47)] + [f"{i:02d}" for i in range(48, 58)],
    "35": [f"{i:02d}" for i in range(17, 27)],  # "17" to "26"
    "38": ["03"]
}

basketball_trials = {
    "06": [f"{i:02d}" for i in range(2, 16)]  # "02" to "15"
}


Shape of X: (135, 93)
type(X):  <class 'numpy.ndarray'>


In [2]:
# copying files to 

import os
import shutil

# --- Source and target paths ---
raw_data_root = r"C:\Users\chenw\OneDrive - University of Pittsburgh\_score network code\Markov chain neumeric\CMU motion capture\raw data"
target_root = r"C:\Users\chenw\OneDrive - University of Pittsburgh\_score network code\Markov chain neumeric\CMU motion capture\training_amc"

# --- Trial dictionaries (subject ID is still unpadded here) ---
running_trials = {
    "02": ["03"],
    "09": [f"{i:02d}" for i in range(1, 12)],
    "16": ["08"] + [f"{i:02d}" for i in range(35, 47)] + [f"{i:02d}" for i in range(48, 58)],
    "35": [f"{i:02d}" for i in range(17, 27)],
    "38": ["03"]
}

basketball_trials = {
    "06": [f"{i:02d}" for i in range(2, 16)]
}

jumping_trials=  {
        "13": ["11", "13", "19", "32", "39", "40", "41", "42"], 
        "16": [f"{i:02d}" for i in range(1, 11)], 
        "49": ["02", "03"], 
    }

def copy_trials(trial_dict, class_name):
    for subject_raw, trial_list in trial_dict.items():
        subject = f"{int(subject_raw):02d}"  # pad to two digits
        source_subject_folder = os.path.join(raw_data_root, f"Subject_{subject}")
        target_subject_folder = os.path.join(target_root, class_name, f"Subject_{subject}")
        os.makedirs(target_subject_folder, exist_ok=True)

        # Copy ASF file
        asf_src = os.path.join(source_subject_folder, f"{subject}.asf")
        asf_dst = os.path.join(target_subject_folder, f"{subject}.asf")
        if os.path.exists(asf_src):
            shutil.copy2(asf_src, asf_dst)
        else:
            print(f"⚠️ Missing ASF: {asf_src}")

        # Copy AMC files
        for trial in trial_list:
            amc_src = os.path.join(source_subject_folder, f"Trial_{trial}.amc")
            amc_dst = os.path.join(target_subject_folder, f"Trial_{trial}.amc")
            if os.path.exists(amc_src):
                shutil.copy2(amc_src, amc_dst)
            else:
                print(f"⚠️ Missing AMC: {amc_src}")

# Run copying
copy_trials(running_trials, class_name="running")
copy_trials(basketball_trials, class_name="basketball")
copy_trials(jumping_trials, class_name="jumping")
print("✅ Done copying trials into training_amc folder.")


✅ Done copying trials into training_amc folder.


In [23]:

root_dir  = r"C:\Users\chenw\OneDrive - University of Pittsburgh\_score network code\Markov chain neumeric\CMU motion capture\training_amc"
# Trial definitions
running_trials = {
    "02": ["03"],
    "09": [f"{i:02d}" for i in range(1, 12)],
    "16": ["08"] + [f"{i:02d}" for i in range(35, 47)] + [f"{i:02d}" for i in range(48, 58)],
    "35": [f"{i:02d}" for i in range(17, 27)],
}

basketball_trials = {
    "06": [f"{i:02d}" for i in range(2, 15)]
}

def count_data_shapes(class_name, trial_dict):
    total_pairs = 0
    print(f"\n===== {class_name.upper()} =====")
    for subject, trials in trial_dict.items():
        subject_dir = os.path.join(root_dir, class_name, f"Subject_{subject}")
        asf_file = os.path.join(subject_dir, f"{subject}.asf")
        for trial in trials:
            amc_file = os.path.join(subject_dir, f"Trial_{trial}.amc")
            try:
                motion = MotionCapture(asf_file, amc_file)
                T, B, _ = motion.data.shape
                D = B * 3
                print(f"Subject {subject}, Trial {trial}: shape = ({T}, {D}) → valid pairs: {T - 1}")
                total_pairs += T - 1
            except Exception as e:
                print(f"⚠️ Failed to load Trial {trial} for Subject {subject}: {e}")
    print(f"Total training pairs in {class_name}: {total_pairs}")
    return total_pairs

if __name__ == "__main__":
    total_running = count_data_shapes("running", running_trials)
    total_basketball = count_data_shapes("basketball", basketball_trials)
    print(f"\nGrand total training pairs: {total_running + total_basketball}")


===== RUNNING =====
Subject 02, Trial 03: shape = (172, 93) → valid pairs: 171
Subject 09, Trial 01: shape = (147, 93) → valid pairs: 146
Subject 09, Trial 02: shape = (129, 93) → valid pairs: 128
Subject 09, Trial 03: shape = (127, 93) → valid pairs: 126
Subject 09, Trial 04: shape = (136, 93) → valid pairs: 135
Subject 09, Trial 05: shape = (142, 93) → valid pairs: 141
Subject 09, Trial 06: shape = (140, 93) → valid pairs: 139
Subject 09, Trial 07: shape = (137, 93) → valid pairs: 136
Subject 09, Trial 08: shape = (127, 93) → valid pairs: 126
Subject 09, Trial 09: shape = (151, 93) → valid pairs: 150
Subject 09, Trial 10: shape = (131, 93) → valid pairs: 130
Subject 09, Trial 11: shape = (164, 93) → valid pairs: 163
Subject 16, Trial 08: shape = (238, 93) → valid pairs: 237
Subject 16, Trial 35: shape = (161, 93) → valid pairs: 160
Subject 16, Trial 36: shape = (188, 93) → valid pairs: 187
Subject 16, Trial 37: shape = (183, 93) → valid pairs: 182
Subject 16, Trial 38: shape = (167,